# ECON62020 - Group Replication Project

## Group
Guy Pigott - 8340277
Trinity Rose
Siqi Ge

## Replication Paper

Leora Friedberg, Elliott Isaac, "Same-Sex Marriage Recognition and Taxes: New Evidence about the Impact of Household Taxation", The Review of Economics and Statistics (2024) 106 (1): 85–101.

## Summary

This project replicates key results from Friedberg and Isaac (2024), which studies how changes in marriage-related tax incentives affect the likelihood of same-sex couples marrying. The central concept is the “marriage subsidy” (or penalty), defined as the difference between the taxes a couple would pay filing jointly versus as two single individuals.

The paper exploits variation in the timing of same-sex marriage recognition across U.S. states, as well as the 2013 federal recognition following United States v. Windsor, to identify the causal impact of tax incentives on marriage behaviour. To isolate exogenous variation, the authors construct a predicted marriage subsidy using LASSO-predicted earnings and use this as an instrument for the observed subsidy.

This replication focuses on reconstructing Table A2 of summary statistics, and reproducing the main results in Table 3, which shows the main OLS and IV regression results with and without controlling for income. The results confirm that higher marriage subsidies increase the probability of being married. While there are some differences in levels due to approximations in the tax simulation, the direction and relative magnitude of the effects are consistent with the original findings.

## Data Description

The analysis uses data derived from the American Community Survey (ACS) for the period 2003–2017. The unit of observation is a same-sex couple, identified from household-level data and classified as either married or cohabiting.

The dataset includes detailed demographic and income information for both partners. Income variables include wages, business income, investment income, retirement income, Social Security income, and government transfers. These are used to construct taxable income inputs for the NBER TAXSIM model.

The main outcome variable is an indicator for whether the couple is married. The key explanatory variable is the marriage subsidy, computed as the difference between joint tax liability and the sum of individual tax liabilities. Two versions are constructed: an observed subsidy based on reported income, and a predicted subsidy based on LASSO-predicted earnings.

Control variables include demographic characteristics (age, education, and differences between partners), household structure (number of children), and indicators for race and gender composition. The specification also includes state and year fixed effects, along with policy variables capturing same-sex marriage recognition and Medicaid expansion.

Due to data limitations, some tax inputs are approximated, which may lead to differences in levels relative to the original paper.

The code first scans a limited set of variables to identify households containing same-sex couples, using partner links (sploc), sex, detailed relationship codes, and a Census recode flag (qrelate). Households are classified as same-sex married or cohabiting, with an additional correction for couples recoded from spouse to unmarried partner.

A second pass reads a richer set of variables only for those households. The sample is restricted to adults aged 18–60 in U.S. states, excluding observations with allocated relationship values. Married and cohabiting couples are then assembled by matching the household head to the relevant partner, and duplicate households are removed.

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

os.chdir("/Users/guypigott/python-venv-demo/EDS")
os.getcwd()

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "Data"
OUTPUT_DIR = PROJECT_ROOT / "Data" / "Output"

In [ ]:
#file_path = "/Users/guypigott/python-venv-demo/Replication Project/Data/Raw/acs_2012-2017.dat"

# Step 1: Read only 5 columns to find (year, serial) of same-sex couple households.
# These columns are sufficient to determine whether a person is in a same-sex couple.
#colspecs_scan = [
 #   (0,   4),    # year
  ##  (6,   14),   # serial
  #  (77,  81),   # pernum
   # (106, 108),  # sploc
   # (126, 130),  # related  (4-digit detailed version)
   # (130, 131),  # sex
   # (269, 270),  # qrelate
#]
#col_names_scan = ["year", "serial", "pernum", "sploc", "related", "sex", "qrelate"]

#print("Step 1: scanning for same-sex couples...")
#df_scan = pd.read_fwf(file_path, colspecs=colspecs_scan, names=col_names_scan)
#print(f"  Total rows: {len(df_scan):,}")

# ── Identify same-sex couples using vectorised operations ──────────────────────────
# Core logic: find partner via sploc, then compare sex values.

# Sort for consistent merging
#df_scan = df_scan.sort_values(["year","serial","pernum"]).reset_index(drop=True)

# Build a lookup table: (year, serial, pernum) → sex / related / qrelate
# Use merge instead of apply (much faster)
#partner_info = df_scan[["year","serial","pernum","sex","related","qrelate"]].copy()
#partner_info.columns = ["year","serial","sploc","partner_sex","partner_related","partner_qrelate"]

##df_scan = df_scan.merge(partner_info, on=["year","serial","sploc"], how="left")

# Same-sex flag
#same_sex    = df_scan["sex"] == df_scan["partner_sex"]
#sploc_valid = df_scan["sploc"] > 0

# Same-sex married: head (101) + spouse (201), or vice versa
#ss_mar = same_sex & sploc_valid & (
   # ((df_scan["related"] == 101)  & (df_scan["partner_related"] == 201)) |
  #  ((df_scan["related"] == 201)  & (df_scan["partner_related"] == 101))
#)

## Same-sex cohabiting: head (101) + unmarried partner (1114), or vice versa
# ss_coh = same_sex & sploc_valid & (
   # ((df_scan["related"] == 101)  & (df_scan["partner_related"] == 1114)) |
   # ((df_scan["related"] == 1114) & (df_scan["partner_related"] == 101))
#)

# mflag correction: qrelate==9 indicates the Census Bureau re-coded a same-sex
# spouse as "unmarried partner"; these should be treated as married.
#mflag = (df_scan["related"] == 1114) & (df_scan["qrelate"] == 9) & ss_coh

# Propagate mflag to the partner as well
#mflag_info = df_scan[["year","serial","pernum"]].copy()
#mflag_info["mflag"] = mflag.values
#mflag_info.columns = ["year","serial","sploc","partner_mflag"]
#df_scan = df_scan.merge(mflag_info, on=["year","serial","sploc"], how="left")
#df_scan["partner_mflag"] = df_scan["partner_mflag"].fillna(False)

#ss_mar = ss_mar | mflag | df_scan["partner_mflag"]
#ss_coh = ss_coh & ~mflag & ~df_scan["partner_mflag"]

#df_scan["ss_all"] = (ss_mar | ss_coh).astype(int)

# Collect the set of (year, serial) keys that belong to SS households
#ss_keys = df_scan.loc[df_scan["ss_all"] == 1, ["year","serial"]].drop_duplicates()
#print(f"  Same-sex couple households found: {len(ss_keys):,}")
#print(f"  Same-sex individuals: {df_scan['ss_all'].sum():,}")

# Core modification: Add the educd (160, 163) field directly to avoid reading the full file twice.
#colspecs_full = [
#    (0, 4), (6, 14), (37, 39), (77, 81), (106, 108), (118, 119), 
#    (124, 126), (126, 130), (130, 131), (131, 134), (134, 135), 
#    (147, 148), (151, 152), (158, 160), (160, 163), (193, 194), 
#    (251, 258), (261, 262), (269, 270)
#]
#col_names_full = [
 #   "year", "serial", "statefip", "pernum", "sploc", "nchild", 
#    "relate", "related", "sex", "age", "marst", 
#    "race", "hispan", "educ", "educd", "uhrswork", 
#    "incearn", "migrate1", "qrelate"
#]

# Create a hash set to speed up lookups
#ss_set = set(zip(ss_keys["year"], ss_keys["serial"]))
#chunks = []
#CHUNK = 500_000

#print("Step 2: reading full columns for SS households only...")
#reader = pd.read_fwf(file_path, colspecs=colspecs_full, names=col_names_full, chunksize=CHUNK)

#for chunk in reader:
#    mask = [(y, s) in ss_set for y, s in zip(chunk["year"], chunk["serial"])]
#   filtered = chunk[mask]
#   if len(filtered) > 0:
#        chunks.append(filtered)

#df_full = pd.concat(chunks, ignore_index=True)


# Calculate the number of adults in the household (used for subsequent precise filtering of cohabiting couples).
#df_full["adult"] = ((df_full["age"] >= 18) | (df_full["sploc"] != 0)).astype(int)
#num_adults = df_full.groupby(["year","serial"])["adult"].sum().reset_index(name="num_adults")
#df_full = df_full.merge(num_adults, on=["year","serial"], how="left")

#print(f"Done. Full data for SS households: {len(df_full):,} rows")

# Sort data to ensure merge logic is accurate
#df_clean = df_full.sort_values(["year","serial","pernum"]).reset_index(drop=True)

# Extract potential partner info and merge it back to the main table
#p_info = df_clean[["year","serial","pernum","sex","related","qrelate","marst"]].copy()
#p_info.columns = ["year","serial","sploc","p_sex","p_related","p_qrelate","p_marst"]
#df_clean = df_clean.merge(p_info, on=["year","serial","sploc"], how="left")

# Basic conditions: Same sex and valid spouse location (sploc)
#same_sex = df_clean["sex"] == df_clean["p_sex"]
#sploc_valid = df_clean["sploc"] > 0

# Identify same-sex married (ss_mar) and same-sex cohabiting (ss_coh)
#ss_mar = same_sex & sploc_valid & (((df_clean["related"] == 101) & (df_clean["p_related"] == 201)) | ((df_clean["related"] == 201) & (df_clean["p_related"] == 101)))
#ss_coh = same_sex & sploc_valid & (((df_clean["related"] == 101) & (df_clean["p_related"] == 1114)) | ((df_clean["related"] == 1114) & (df_clean["p_related"] == 101)))

# Handle mflag (married couples identified by qrelate==9)
#mflag = (df_clean["related"] == 1114) & (df_clean["qrelate"] == 9) & ss_coh
#mflag_info = df_clean[["year","serial","pernum"]].copy()
#mflag_info["mflag"] = mflag.values
#mflag_info.columns = ["year","serial","sploc","p_mflag"]

#df_clean = df_clean.merge(mflag_info, on=["year","serial","sploc"], how="left")
#df_clean["p_mflag"] = df_clean["p_mflag"].fillna(False)


# Finalize marriage and cohabitation classification
#ss_mar_final = ss_mar | mflag | df_clean["p_mflag"]
#ss_coh_final = ss_coh & ~mflag & ~df_clean["p_mflag"]

#df_clean["sscouple_mar"] = ss_mar_final.astype(int)
#df_clean["sscouple_coh"] = ss_coh_final.astype(int)
#df_clean["ss_any"] = (ss_mar_final | ss_coh_final).astype(int)

# Base filters: Age 18-60, exclude territories, exclude cases with allocated relationships
#mask_base = (
#    (df_clean["ss_any"] == 1) & 
#    (df_clean["age"] >= 18) & (df_clean["age"] <= 60) & 
#    (df_clean["statefip"] <= 56) & 
#    (df_clean["qrelate"] != 4) & (df_clean.get("p_qrelate", 0) != 4)
#)
#df_valid = df_clean[mask_base].copy()

# ================= Assemble Samples =================
# Married logic: Head (101) and their spouse
# mc = df_valid[df_valid["sscouple_mar"] == 1].copy()
#head_m = mc[mc["related"] == 101]
#spouse_m = mc[mc["related"].isin([201, 1114])]
#spouse_m = spouse_m.sort_values("pernum").groupby(["year","serial"]).first().reset_index()
#df_mar = head_m.merge(spouse_m, on=["year","serial"], suffixes=("_h","_s"))
#df_mar["married"] = 1

# Cohabiting logic: To reach 21,136, we typically do not restrict num_adults
#cc = df_valid[df_valid["sscouple_coh"] == 1].copy()
#head_c = cc[cc["related"] == 101]
#partner_c = cc[cc["related"] == 1114]
#df_coh = head_c.merge(partner_c, on=["year","serial"], suffixes=("_h","_s"))
#df_coh["married"] = 0

# Ensure bidirectional sploc matching (excludes complex cohabitation structures)
#df_coh = df_coh[(df_coh["sploc_h"] == df_coh["pernum_s"]) & (df_coh["sploc_s"] == df_coh["pernum_h"])]

# Concatenate and drop duplicates
#df_final = pd.concat([df_mar, df_coh], axis=0, ignore_index=True)
#df_final = df_final.drop_duplicates(subset=["year", "serial"])

#print(f"=== Final Sample (Target Alignment) ===")
#print(f"Married:    {len(df_final[df_final['married']==1]):,} (Paper: 16,098)")
#print(f"Cohabiting: {len(df_final[df_final['married']==0]):,} (Paper: 21,136)")

#Save Cleaned Data
#output_dir = Path("/Users/guypigott/python-venv-demo/EDS/Data/Output")
#output_dir.mkdir(parents=True, exist_ok=True)

#cleaned_file = output_dir / "cleaned_data.pkl"
##df_final.to_pickle(cleaned_file)

#print(f"Saved cleaned data to: {cleaned_file}")
#print(f"Shape: {df_final.shape}")

#csv_file = output_dir / "cleaned_data.csv"
#df_final.to_csv(csv_file, index=False)
#print(f"Saved CSV copy to: {csv_file}")

The final preparation stage constructs the variables used in estimation: partner earnings, total couple earnings, earnings split, age and education measures, race matching, children, and employment structure. Education is converted from detailed ACS codes into years of schooling, and policy variables are added for state marriage recognition, Windsor/Obergefell timing, and Medicaid expansion. The cleaned couple-level dataset is then saved for the later LASSO, TAXSIM, and regression stages.

In [8]:
cleaned_data_path = Path("/Users/guypigott/python-venv-demo/EDS/Data/Output/cleaned_data.pkl")

df_final = pd.read_pickle(cleaned_data_path)

# Vectorized conversion: educd (detailed education code) to years of education
def educd_to_yrs_vec(s):
    s = s.fillna(-1).astype(int)
    result = np.full(len(s), np.nan)
    result[s <= 12] = 0
    result[s.isin([14, 15, 16, 17])] = s[s.isin([14, 15, 16, 17])] - 13
    result[s.isin([22, 23])] = s[s.isin([22, 23])] - 17
    result[s.isin([25, 26])] = s[s.isin([25, 26])] - 18
    result[s == 30] = 9
    result[s == 40] = 10
    result[s == 50] = 11
    result[(s >= 61) & (s <= 65)] = 12
    result[s == 71] = 13
    result[s == 81] = 14
    result[s == 101] = 16
    result[s >= 114] = 18
    return result

df_f = df_final.copy()

In [13]:
# Income and earnings split calculation
df_f["earn_h"] = df_f["incearn_h"].fillna(0).clip(lower=0)
df_f["earn_s"] = df_f["incearn_s"].fillna(0).clip(lower=0)
df_f["total_earn"] = df_f["earn_h"] + df_f["earn_s"]
df_f["earn_split"] = np.where(df_f["total_earn"] > 0, df_f[["earn_h","earn_s"]].max(axis=1) / df_f["total_earn"], np.nan)

# Construction of demographics and employment status variables
df_f["male"] = (df_f["sex_h"] == 1).astype(int)
df_f["female"] = (df_f["sex_h"] == 2).astype(int)
df_f["age_old"] = np.maximum(df_f["age_h"], df_f["age_s"])
df_f["age_yng"] = np.minimum(df_f["age_h"], df_f["age_s"])
df_f["age_diff"] = df_f["age_old"] - df_f["age_yng"]

df_f["both_work"] = ((df_f["earn_h"]>0) & (df_f["earn_s"]>0)).astype(int)
df_f["one_work"] = ((df_f["earn_h"]>0) ^ (df_f["earn_s"]>0)).astype(int)
df_f["none_work"] = ((df_f["earn_h"]==0) & (df_f["earn_s"]==0)).astype(int)

# Racial matching and dependent children identification
df_f["same_race"] = ((df_f["race_h"] == df_f["race_s"]) & ((df_f["hispan_h"] > 0) == (df_f["hispan_s"] > 0))).astype(int)
df_f["any_children"] = (df_f["nchild_h"] > 0).astype(int)

# Application of education years
df_f["edu_h_yrs"] = educd_to_yrs_vec(df_f["educd_h"])
df_f["edu_s_yrs"] = educd_to_yrs_vec(df_f["educd_s"])
df_f["edu_max"] = np.maximum(df_f["edu_h_yrs"], df_f["edu_s_yrs"])
df_f["edu_min"] = np.minimum(df_f["edu_h_yrs"], df_f["edu_s_yrs"])
df_f["edu_diff"] = df_f["edu_max"] - df_f["edu_min"]


# Output results comparison table
m = df_f[df_f["married"]==1]
c = df_f[df_f["married"]==0]

In [15]:
# ── helper maps ──────────────────────────────────────────────────
def map_education_group(yrs):
    out = np.full(len(yrs), np.nan)
    out[yrs <  12] = 1
    out[yrs == 12] = 2
    out[(yrs > 12) & (yrs < 16)] = 3
    out[yrs >= 16] = 4
    return out

def map_age_group(age):
    out = np.full(len(age), np.nan)
    for upper in range(20, 105, 5):
        mask = (age <= upper) & (age > upper - 5)
        out[mask] = upper
    return out

def map_race(race, hispan):
    out = np.full(len(race), np.nan)
    out[(race == 2) & (hispan == 0)] = 1
    out[((race >= 4) & (race <= 6)) & (hispan == 0)] = 2
    out[((race == 3) | (race >= 7)) & (hispan == 0)] = 3
    out[(race == 1) & (hispan == 0)] = 4
    out[hispan > 0] = 5
    return out

def educd_to_yrs(s):
    s = pd.array(s, dtype="Int64")
    result = np.full(len(s), np.nan)
    result[s <= 12] = 0
    result[s == 14] = 1
    result[s == 15] = 2
    result[s == 16] = 3
    result[s == 17] = 4
    result[s == 22] = 5
    result[s == 23] = 6
    result[s == 25] = 7
    result[s == 26] = 8
    result[s == 30] = 9
    result[s == 40] = 10
    result[s == 50] = 11
    result[(s >= 61) & (s <= 65)] = 12
    result[s == 71] = 13
    result[s == 81] = 14
    result[s == 101] = 16
    result[s >= 114] = 18
    return result

# ── build output dataframe ───────────────────────────────────────
d = df_final.copy()
d["uhrswork_h"] = pd.to_numeric(d["uhrswork_h"], errors="coerce").fillna(0)
d["uhrswork_s"] = pd.to_numeric(d["uhrswork_s"], errors="coerce").fillna(0)
d["incearn_h"] = d["incearn_h"].clip(lower=0)
d["incearn_s"] = d["incearn_s"].clip(lower=0)

out = pd.DataFrame()
out["year"]         = d["year"]
out["serial"]       = d["serial"]
out["statefip"]     = d["statefip_h"]
out["sscouple_mar"] = d["married"].astype(bool)
out["sscouple_coh"] = (d["married"] == 0).astype(bool)
out["sscouple_all"] = True
out["r_male"]       = (d["sex_h"] == 1).astype(int)
out["age"]          = d["age_h"]
out["r_incearn"]    = d["incearn_h"]
r_yrs               = educd_to_yrs(d["educd_h"].fillna(-1).astype(int).values)
out["r_yrsedu"]     = r_yrs
out["r_edugroup"]   = map_education_group(r_yrs)
out["r_agegroup"]   = map_age_group(d["age_h"].values)
out["r_race"]       = map_race(d["race_h"].values, d["hispan_h"].values)
out["r_occbroad"]   = 0
out["r_degbroad"]   = 0
out["sp_male"]      = (d["sex_s"] == 1).astype(int)
out["sp_age"]       = d["age_s"]
out["sp_incearn"]   = d["incearn_s"]
sp_yrs              = educd_to_yrs(d["educd_s"].fillna(-1).astype(int).values)
out["sp_yrsedu"]    = sp_yrs
out["sp_edugroup"]  = map_education_group(sp_yrs)
out["sp_agegroup"]  = map_age_group(d["age_s"].values)
out["sp_race"]      = map_race(d["race_s"].values, d["hispan_s"].values)
out["sp_occbroad"]  = 0
out["sp_degbroad"]  = 0
out["c_children"]   = d["nchild_h"]
out["c_anychildren"]= (d["nchild_h"] > 0).astype(int)
out["c_racecomp"]   = (out["r_race"] == out["sp_race"]).astype(int)
out["c_agemax"]     = np.maximum(d["age_h"], d["age_s"])
out["c_agemin"]     = np.minimum(d["age_h"], d["age_s"])
out["c_agediff"]    = out["c_agemax"] - out["c_agemin"]
out["c_edumax"]     = np.maximum(r_yrs, sp_yrs)
out["c_edumin"]     = np.minimum(r_yrs, sp_yrs)
out["c_edudiff"]    = out["c_edumax"] - out["c_edumin"]
out["c_incearn"]    = d["incearn_h"] + d["incearn_s"]
total               = out["c_incearn"].replace(0, np.nan)
out["c_incearn_split"] = (np.maximum(d["incearn_h"], d["incearn_s"]) / total).fillna(0.5)
for i in range(1, 6):
    out[f"c_incearn{i}"]       = (out["c_incearn"] / 10_000) ** i
    out[f"c_incearn_split{i}"] =  out["c_incearn_split"] ** i
out["r_posincearn"]  = (d["incearn_h"] > 0).astype(int)
out["sp_posincearn"] = (d["incearn_s"] > 0).astype(int)
out["c_dualearner"]  = ((d["incearn_h"] > 0) & (d["incearn_s"] > 0)).astype(int)
out["c_singleearner"]= ((d["incearn_h"] > 0) ^ (d["incearn_s"] > 0)).astype(int)
out["c_noearner"]    = ((d["incearn_h"] == 0) & (d["incearn_s"] == 0)).astype(int)
out["migrate1"]      = d["migrate1_h"]

# ── policy columns ───────────────────────────────────────────────
ssm_start = {
    2: 2014, 4: 2014, 6: 2013, 8: 2014, 9: 2008, 10: 2013, 11: 2010,
    12: 2015, 15: 2013, 16: 2014, 17: 2014, 18: 2014, 19: 2009,
    23: 2012, 24: 2013, 25: 2004, 27: 2013, 30: 2014, 32: 2014,
    33: 2010, 34: 2013, 35: 2013, 36: 2011, 37: 2014, 40: 2014,
    41: 2014, 42: 2014, 44: 2013, 45: 2014, 49: 2014, 50: 2009,
    51: 2014, 53: 2012, 54: 2014, 55: 2014, 56: 2014
}
out["staterecog_policy"] = (
    out["year"] >= out["statefip"].map(ssm_start).fillna(2015)
).astype(int)
out["preW"]      = (out["year"] <= 2012).astype(int)
out["postWpreO"] = (out["year"].between(2013, 2014)).astype(int)
out["postO"]     = (out["year"] >= 2015).astype(int)

MEDICAID_EXP_STATES = {
    4, 6, 8, 9, 10, 11, 15, 17, 18, 19, 21, 23, 24, 25,
    26, 27, 32, 33, 34, 35, 36, 38, 41, 44, 50, 53, 55
}
out["medicaid_exp"] = (
    (out["year"] >= 2014) & (out["statefip"].isin(MEDICAID_EXP_STATES))
).astype(int)

# ── sanity check & save ──────────────────────────────────────────
print(f"Output shape: {out.shape}")
print(f"Policy columns: {['staterecog_policy','preW','postWpreO','postO','medicaid_exp']}")
out.to_pickle("/Users/guypigott/python-venv-demo/EDS/Data/Output/acs_ssc_lasso_input_final.pkl")

Output shape: (37907, 56)
Policy columns: ['staterecog_policy', 'preW', 'postWpreO', 'postO', 'medicaid_exp']


# Table A2 

Table A2 reports summary statistics for the sample of same-sex couples used in the analysis, split between married and cohabiting couples. The table is designed to demonstrate that the constructed dataset closely matches the sample and descriptive statistics reported in the original paper.

In [14]:
print("=== Table A2 Full Comparison ===\n")
print(f"{'Variable':<35} {'Our Married':>18} {'Our Cohab':>18} {'Paper Married':>14} {'Paper Cohab':>12}")
print("─"*100)

def row(label, s1, s2, pm, pc):
    v1 = f"{np.nanmean(s1):.3f} ({np.nanstd(s1):.3f})"
    v2 = f"{np.nanmean(s2):.3f} ({np.nanstd(s2):.3f})"
    print(f"{label:<35} {v1:>18} {v2:>18} {pm:>14} {pc:>12}")

row("Male", m["male"], c["male"], "0.469 (0.499)", "0.506 (0.500)")
row("Female", m["female"], c["female"], "0.531 (0.499)", "0.494 (0.500)")
row("Same race", m["same_race"], c["same_race"], "0.793 (0.405)", "0.757 (0.429)")
row("Age older", m["age_old"], c["age_old"], "46.205 (9.633)","43.353 (10.902)")
row("Age younger", m["age_yng"], c["age_yng"], "41.252 (9.831)","37.858 (10.442)")
row("Age diff", m["age_diff"], c["age_diff"], "4.953 (5.234)", "5.495 (5.543)")
row("Edu max (yrs)", m["edu_max"], c["edu_max"], "15.594 (2.468)","15.363 (2.264)")
row("Edu min (yrs)", m["edu_min"], c["edu_min"], "13.718 (3.044)","13.513 (2.604)")
row("Edu diff", m["edu_diff"], c["edu_diff"], "1.875 (2.286)", "1.850 (2.108)")
row("Any children", m["any_children"], c["any_children"], "0.314 (0.464)","0.162 (0.369)")
row("Both work", m["both_work"], c["both_work"], "0.776 (0.417)", "0.811 (0.392)")
row("Only 1 works", m["one_work"], c["one_work"], "0.195 (0.396)", "0.160 (0.367)")
row("Neither works", m["none_work"], c["none_work"], "0.029 (0.167)", "0.029 (0.168)")
row("Total earnings", m["total_earn"], c["total_earn"], "125287 (119780)","105188 (105192)")
row("Earnings split", m.loc[m["total_earn"]>0,"earn_split"], c.loc[c["total_earn"]>0,"earn_split"], "0.745 (0.200)", "0.723 (0.174)")
print("─"*100)
print(f"{'Observations':<35} {len(m):>18,} {len(c):>18,} {'16,098':>14} {'21,136':>12}")

=== Table A2 Full Comparison ===

Variable                                   Our Married          Our Cohab  Paper Married  Paper Cohab
────────────────────────────────────────────────────────────────────────────────────────────────────
Male                                     0.468 (0.499)      0.497 (0.500)  0.469 (0.499) 0.506 (0.500)
Female                                   0.532 (0.499)      0.503 (0.500)  0.531 (0.499) 0.494 (0.500)
Same race                                0.783 (0.412)      0.744 (0.436)  0.793 (0.405) 0.757 (0.429)
Age older                               46.209 (9.634)    43.251 (10.933) 46.205 (9.633) 43.353 (10.902)
Age younger                             41.241 (9.842)    37.698 (10.462) 41.252 (9.831) 37.858 (10.442)
Age diff                                 4.968 (5.263)      5.554 (5.690)  4.953 (5.234) 5.495 (5.543)
Edu max (yrs)                           15.588 (2.472)     15.345 (2.264) 15.594 (2.468) 15.363 (2.264)
Edu min (yrs)                        

Overall, the replication closely matches the original results across all key variables.

The sample sizes are nearly identical to those in the paper, with 16,146 married couples and 21,761 cohabiting couples, compared to 16,098 and 21,136 respectively. The gender, race and age distributions, and education levels align almost perfectly with the original table.

Labour supply and income measures are also well replicated. Average total earnings for married couples are £125,125 in our sample compared to £125,287 in the paper, while cohabiting couples show equally close correspondence. The within-couple earnings split is also replicated almost exactly, with values of 0.745 for married couples and 0.722 for cohabiting couples, matching the published results.

The only noticeable deviations are small differences in the share of couples with children, where our estimates are slightly higher (0.334 vs 0.314 for married couples). These differences are minor and likely reflect small discrepancies in variable construction or sample filtering.

Overall, the close match across demographic characteristics, labour supply outcomes, and income measures indicates that the data cleaning and sample construction successfully reproduce the dataset used in the original paper. This provides a strong foundation for the subsequent LASSO predictions and TAXSIM-based subsidy calculations.

The near-exact replication of earnings and labour supply measures is particularly important, as these variables are central to the LASSO prediction and TAXSIM simulations used in the subsequent analysis.